# Annual National Feature for Syria

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(marker="pyproject.toml"):
    """Walk up from this notebook's directory until we find the project root."""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "syria-nowcasting"

RAW_GLOB = DATA_DIR / "raw"
BOUNDARY_PATH = DATA_DIR / "boundaries" / "syria_adm0.geojson"
OUT_PATH = DATA_DIR / "processed" / "annual_gdp_proxy_features.csv"
POP_PATH = DATA_DIR / "processed" / "quadkey_population_annual.parquet"
SPEED_TIER_OUT_PATH = DATA_DIR / "processed" / "annual_speed_tier_population.csv"

In [2]:
def load_tiles() -> pd.DataFrame:
    """Concatenate every quarterly tile file into one long frame."""
    frames = []
    for f in sorted(RAW_GLOB.glob("**/*.parquet")):
        try:
            df = pd.read_parquet(
                f,
                columns=[
                    "quadkey",
                    "avg_d_kbps",
                    "avg_u_kbps",
                    "avg_lat_ms",
                    "tests",
                    "devices",
                    "quarter",
                    "type",
                    "year",
                ],
            )
        except Exception:
            continue
        if df.empty:
            continue
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


tiles = load_tiles()
tiles.head()


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,quarter,type,year
0,1221103222122032,159,2685,71,4,1,1,fixed,2019
1,1221103222123001,7401,6838,28,1,1,1,fixed,2019
2,1221103222131220,809,768,78,1,1,1,fixed,2019
3,1221103222212232,10816,11799,35,12,1,1,fixed,2019
4,1221103222230233,4574,1974,6,2,1,1,fixed,2019


In [3]:
def test_weighted_aggregate(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Collapse `df` to one row per `group_cols`, test-weighting speed/latency."""
    x = df.assign(
        d_x_tests=df["avg_d_kbps"] * df["tests"],
        u_x_tests=df["avg_u_kbps"] * df["tests"],
        lat_x_tests=df["avg_lat_ms"] * df["tests"],
    )
    agg = x.groupby(group_cols, as_index=False).agg(
        tests=("tests", "sum"),
        devices=("devices", "sum"),
        d_x_tests=("d_x_tests", "sum"),
        u_x_tests=("u_x_tests", "sum"),
        lat_x_tests=("lat_x_tests", "sum"),
    )
    agg["avg_d_kbps"] = agg["d_x_tests"] / agg["tests"]
    agg["avg_u_kbps"] = agg["u_x_tests"] / agg["tests"]
    agg["avg_lat_ms"] = agg["lat_x_tests"] / agg["tests"]
    return agg.drop(columns=["d_x_tests", "u_x_tests", "lat_x_tests"])


def pool_tiles_across_type(tiles: pd.DataFrame) -> pd.DataFrame:
    """Pool fixed + mobile into a `type="combined"` pseudo-type, per (year, quarter, quadkey)."""
    pooled = test_weighted_aggregate(tiles, ["year", "quarter", "quadkey"])
    pooled.insert(0, "type", "combined")
    return pooled[tiles.columns]


tiles = pd.concat([tiles, pool_tiles_across_type(tiles)], ignore_index=True)
tiles.sample(5, random_state=10027)


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,quarter,type,year
54746,1221120131031002,816.000000,848.000000,60.000000,41,1,1,fixed,2022
539929,1221121000213200,2425.166667,4352.833333,48.666667,6,5,2,combined,2024
659156,1221121220033211,39397.348837,12746.604651,35.186047,43,10,4,combined,2025
475258,1221121002233222,1097.000000,4447.000000,67.000000,2,2,4,combined,2022
64997,1221120123330102,5034.000000,1862.000000,52.000000,60,28,2,fixed,2022


## Annual aggregation

For each `type` (`fixed`, `mobile`, or `combined`) and `year`:

- `total_tests`, `total_devices_period`: summed across the year's quarters. Note that `total_devices_period` sums each quarter's unique devices in that quarter, so it's device-quarters, not unique annual devices. Thus a device active all four quarters is counted four times.
- `n_tiles_active`: count of distinct tiles (`quadkey`) with any test activity that year.
- `coverage_share`: `n_tiles_active` divided by the number of distinct tiles ever observed for that type across the entire dataset (2019–present).
- `avg_down_mbps` / `avg_up_mbps` / `avg_lat_ms`: test-weighted mean across all tile-quarter rows in the year (weight = `tests`), so a quarter with more testing activity contributes proportionally more to the annual figure.

In [4]:
def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    return float((values * weights).sum() / weights.sum())


def build_annual_panel(tiles: pd.DataFrame) -> pd.DataFrame:
    universe_size = (
        tiles.groupby("type")["quadkey"].nunique().rename("n_tiles_universe")
    )

    rows = []
    for (t, y), g in tiles.groupby(["type", "year"]):
        rows.append(
            dict(
                type=t,
                year=int(y),
                total_tests=int(g["tests"].sum()),
                total_devices_period=int(g["devices"].sum()),
                n_tiles_active=g["quadkey"].nunique(),
                avg_down_mbps=round(
                    weighted_mean(g["avg_d_kbps"], g["tests"]) / 1000, 3
                ),
                avg_up_mbps=round(weighted_mean(g["avg_u_kbps"], g["tests"]) / 1000, 3),
                avg_lat_ms=round(weighted_mean(g["avg_lat_ms"], g["tests"]), 1),
                n_quarters_observed=g["quarter"].nunique(),
            )
        )
    panel = pd.DataFrame(rows).merge(universe_size, on="type")
    panel["coverage_share"] = round(
        panel["n_tiles_active"] / panel["n_tiles_universe"], 4
    )

    return panel.sort_values(["type", "year"]).reset_index(drop=True)


annual_features = build_annual_panel(tiles)
# annual_features.to_csv(OUT_PATH, index=False)
# print(f"Saved -> {OUT_PATH}")
annual_features


,type,year,total_tests,total_devices_period,n_tiles_active,avg_down_mbps,avg_up_mbps,avg_lat_ms,n_quarters_observed,n_tiles_universe,coverage_share
0,combined,2019,440857,118243,16815,7.847,8.925,58.2,4,62471,0.2692
1,combined,2020,138966,43961,8101,9.401,6.448,63.4,4,62471,0.1297
2,combined,2021,373884,106955,15135,11.836,11.970,38.8,4,62471,0.2423
3,combined,2022,1139918,275750,20642,11.450,14.756,32.7,4,62471,0.3304
4,combined,2023,932894,251631,19583,13.281,18.476,30.0,4,62471,0.3135
5,combined,2024,935309,258804,20692,15.316,19.344,34.8,4,62471,0.3312
6,combined,2025,2109652,556778,44933,16.830,11.848,62.6,4,62471,0.7193
7,fixed,2019,409878,103798,15278,6.874,8.906,57.2,4,55865,0.2735
8,fixed,2020,103242,24550,4688,5.895,5.507,68.9,4,55865,0.0839
9,fixed,2021,311919,79409,12455,10.371,12.601,36.7,4,55865,0.2229


## Population & connectivity features

The annual panel above weights every quarter by test volume, which tell us about speeds where people happened to run tests, but says nothing about how many people actually live in covered vs uncovered areas, or what tier of service they experience. This section joins a population layer to the Ookla tile grid so we can express coverage and speed as a share of people, not tiles.

**Data source**: [WorldPop](https://hub.worldpop.org/geodata/listing?id=135)
constrained global mosaic, Syria, one raster per year (2015-2030 series, `R2025A`) which maps each populated pixel to its zoom-16 quadkey (verified against the `tile_x`/`tile_y` columns present in Ookla files from Q3 2023 onward), and sums population per quadkey for that year.

**Caveats**:

- This series' Syria total climbs from ~19.96M (2019) to ~25.17M (2025), a smooth upward trend that does not capture the sharp wartime outflows and post-2024 return movements. WorldPop's constrained layers redistribute a demographically-projected national total onto built-up pixels, thus they are not a real-time census or displacement count.
- Constrained layers restrict population to built-up-area pixels (Random Forest dasymetric redistribution against a settlement mask), producing a sparser, more concentrated pattern than the smoother unconstrained layer.
- It counts unique devices that ran an Ookla speed test in a quadkey that quarter, which is a small, self-selected sample of people who chose to run a test, not a census of internet users. The metrics below are framed as population living in tiles with observed test activity, not population "online".


In [5]:
population = pd.read_parquet(POP_PATH)
TOTAL_POPULATION_BY_YEAR = population.groupby("year")["population"].sum()
population.head()


,year,quadkey,population
0,2019,1221103222012213,1.3
1,2019,1221103222012300,127.2
2,2019,1221103222012301,81.8
3,2019,1221103222012302,559.3
4,2019,1221103222012303,245.1


In [6]:
def build_quadkey_year_panel(tiles: pd.DataFrame) -> pd.DataFrame:
    """Collapse quarters to one row per (type, year, quadkey), test-weighted."""
    return test_weighted_aggregate(tiles, ["type", "year", "quadkey"])


quadkey_year_panel = build_quadkey_year_panel(tiles).merge(
    population, on=["year", "quadkey"], how="left"
)
quadkey_year_panel["population"] = quadkey_year_panel["population"].fillna(0)
quadkey_year_panel.head()


,type,year,quadkey,tests,devices,avg_d_kbps,avg_u_kbps,avg_lat_ms,population
0,combined,2019,1221103222012300,25,6,7670.160000,8114.240000,57.080000,127.2
1,combined,2019,1221103222012301,10,4,6598.400000,7333.000000,72.600000,81.8
2,combined,2019,1221103222012302,55,18,5524.472727,6359.890909,59.036364,559.3
3,combined,2019,1221103222012303,3,3,6011.000000,6197.000000,37.000000,245.1
4,combined,2019,1221103222012320,4,3,1744.750000,4178.500000,69.000000,158.2


In [7]:
def population_weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    if weights.sum() <= 0:
        return float("nan")
    return weighted_mean(values, weights)


def build_population_features(
    quadkey_year: pd.DataFrame,
    total_population_by_year: pd.Series,
) -> pd.DataFrame:
    rows = []
    for (t, y), g in quadkey_year.groupby(["type", "year"]):
        active = g[g["tests"] > 0]
        pop_covered = active["population"].sum()
        total_population = total_population_by_year.get(y, np.nan)

        rows.append(
            dict(
                type=t,
                year=int(y),
                pop_covered=round(pop_covered, 0),
                pop_covered_share=round(pop_covered / total_population, 4)
                if total_population
                else np.nan,
                device_periods_per_1k_pop_covered=round(
                    1000 * active["devices"].sum() / pop_covered, 3
                )
                if pop_covered > 0
                else np.nan,
                avg_down_mbps_pop_weighted=round(
                    population_weighted_mean(active["avg_d_kbps"], active["population"])
                    / 1000,
                    3,
                ),
                avg_up_mbps_pop_weighted=round(
                    population_weighted_mean(active["avg_u_kbps"], active["population"])
                    / 1000,
                    3,
                ),
            )
        )
    return pd.DataFrame(rows).sort_values(["type", "year"]).reset_index(drop=True)


population_features = build_population_features(
    quadkey_year_panel, TOTAL_POPULATION_BY_YEAR
)
population_features


,type,year,pop_covered,pop_covered_share,device_periods_per_1k_pop_covered,avg_down_mbps_pop_weighted,avg_up_mbps_pop_weighted
0,combined,2019,10909450.0,0.5467,10.839,7.190,4.999
1,combined,2020,9322324.0,0.4492,4.716,10.431,5.495
2,combined,2021,11374569.0,0.5328,9.403,9.679,6.262
3,combined,2022,12944399.0,0.5908,21.303,8.540,6.261
4,combined,2023,12889372.0,0.5601,19.522,11.640,7.743
5,combined,2024,14639455.0,0.6056,17.679,12.388,6.800
6,combined,2025,21029818.0,0.8355,26.476,17.504,8.657
7,fixed,2019,10011015.0,0.5016,10.368,3.891,3.959
8,fixed,2020,7079662.0,0.3412,3.468,4.111,3.495
9,fixed,2021,9571309.0,0.4483,8.297,5.136,4.576


In [8]:
annual_features = annual_features.merge(
    population_features, on=["type", "year"], how="left"
)
annual_features.to_csv(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")
annual_features


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/annual_gdp_proxy_features.csv


,type,year,total_tests,total_devices_period,n_tiles_active,avg_down_mbps,avg_up_mbps,avg_lat_ms,n_quarters_observed,n_tiles_universe,coverage_share,pop_covered,pop_covered_share,device_periods_per_1k_pop_covered,avg_down_mbps_pop_weighted,avg_up_mbps_pop_weighted
0,combined,2019,440857,118243,16815,7.847,8.925,58.2,4,62471,0.2692,10909450.0,0.5467,10.839,7.190,4.999
1,combined,2020,138966,43961,8101,9.401,6.448,63.4,4,62471,0.1297,9322324.0,0.4492,4.716,10.431,5.495
2,combined,2021,373884,106955,15135,11.836,11.970,38.8,4,62471,0.2423,11374569.0,0.5328,9.403,9.679,6.262
3,combined,2022,1139918,275750,20642,11.450,14.756,32.7,4,62471,0.3304,12944399.0,0.5908,21.303,8.540,6.261
4,combined,2023,932894,251631,19583,13.281,18.476,30.0,4,62471,0.3135,12889372.0,0.5601,19.522,11.640,7.743
5,combined,2024,935309,258804,20692,15.316,19.344,34.8,4,62471,0.3312,14639455.0,0.6056,17.679,12.388,6.800
6,combined,2025,2109652,556778,44933,16.830,11.848,62.6,4,62471,0.7193,21029818.0,0.8355,26.476,17.504,8.657
7,fixed,2019,409878,103798,15278,6.874,8.906,57.2,4,55865,0.2735,10011015.0,0.5016,10.368,3.891,3.959
8,fixed,2020,103242,24550,4688,5.895,5.507,68.9,4,55865,0.0839,7079662.0,0.3412,3.468,4.111,3.495
9,fixed,2021,311919,79409,12455,10.371,12.601,36.7,4,55865,0.2229,9571309.0,0.4483,8.297,5.136,4.576


### Any-type coverage: fixed, mobile, both, or neither

`pop_covered_share` above is computed separately per `type`, so it can't answer "what share of the population has *any* connectivity at all," since someone could show up as uncovered in the `fixed` row while actually having mobile service (or vice versa). This section classifies each populated quadkey into exactly one of `fixed_only`, `mobile_only`, `both`, or `neither` for each year, so the shares sum to 1 by construction.


In [9]:
ANY_TYPE_COVERAGE_OUT_PATH = DATA_DIR / "processed" / "annual_any_type_coverage.csv"


def build_any_type_coverage(
    quadkey_year: pd.DataFrame,
    population: pd.DataFrame,
    total_population_by_year: pd.Series,
) -> pd.DataFrame:
    # "combined" is excluded here on purpose: this table's whole point is to
    # distinguish fixed_only / mobile_only / both, which the pooled type can't do.
    active = quadkey_year.loc[
        (quadkey_year["tests"] > 0) & (quadkey_year["type"] != "combined"),
        ["type", "year", "quadkey"],
    ]
    pivot = (
        active.assign(active=True)
        .pivot_table(
            index=["year", "quadkey"], columns="type", values="active", fill_value=0
        )
        .astype(bool)
        .reset_index()
    )
    for col in ("fixed", "mobile"):
        if col not in pivot.columns:
            pivot[col] = False

    pivot["category"] = np.select(
        [
            pivot["fixed"] & pivot["mobile"],
            pivot["fixed"] & ~pivot["mobile"],
            ~pivot["fixed"] & pivot["mobile"],
        ],
        ["both", "fixed_only", "mobile_only"],
        default="unreachable",  # every row here has at least one type active, so this never fires
    )
    pivot = pivot.merge(population, on=["year", "quadkey"], how="left")
    pivot["population"] = pivot["population"].fillna(0)
    by_category = (
        pivot.groupby(["year", "category"])["population"]
        .sum()
        .unstack("category", fill_value=0.0)
    )
    for col in ("fixed_only", "mobile_only", "both"):
        if col not in by_category.columns:
            by_category[col] = 0.0
    by_category["total_population"] = by_category.index.map(total_population_by_year)
    by_category["neither"] = by_category["total_population"] - by_category[
        ["fixed_only", "mobile_only", "both"]
    ].sum(axis=1)

    shares = by_category.div(by_category["total_population"], axis=0).drop(
        columns="total_population"
    )
    shares = shares.rename(columns={c: f"pop_share_{c}" for c in shares.columns})
    shares["pop_covered_share_any_type"] = round(
        shares[["pop_share_fixed_only", "pop_share_mobile_only", "pop_share_both"]].sum(
            axis=1
        ),
        4,
    )
    return shares.round(4).reset_index()


any_type_coverage = build_any_type_coverage(
    quadkey_year_panel, population, TOTAL_POPULATION_BY_YEAR
)
any_type_coverage.to_csv(ANY_TYPE_COVERAGE_OUT_PATH, index=False)
print(f"Saved -> {ANY_TYPE_COVERAGE_OUT_PATH}")
any_type_coverage

Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/annual_any_type_coverage.csv


category,year,pop_share_both,pop_share_fixed_only,pop_share_mobile_only,pop_share_neither,pop_covered_share_any_type
0,2019,0.3116,0.1901,0.0450,0.4533,0.5467
1,2020,0.2737,0.0674,0.1081,0.5508,0.4492
2,2021,0.3501,0.0983,0.0845,0.4672,0.5328
3,2022,0.4014,0.1056,0.0838,0.4092,0.5908
4,2023,0.3575,0.0646,0.1380,0.4399,0.5601
5,2024,0.3935,0.0868,0.1253,0.3944,0.6056
6,2025,0.6236,0.1828,0.0291,0.1645,0.8355


### Population by service-speed tier

We use the FCC National Broadband Map tiers (updated March 2024) to define the speed tier. A tile only qualifies for a tier if it clears both thresholds:

| Tier | Download | Upload |
|---|---|---|
| Below minimum | < 25 Mbps | < 3 Mbps |
| Mid (25/3) | ≥ 25 Mbps | ≥ 3 Mbps |
| Meets current US broadband standard (100/20) | ≥ 100 Mbps | ≥ 20 Mbps |
| Gigabit (1000/100) | ≥ 1000 Mbps | ≥ 100 Mbps |

Sources: [FCC broadband speed benchmark increase, March 2024](https://docs.fcc.gov/public/attachments/DOC-401205A1.pdf),
[FCC National Broadband Map](https://broadbandmap.fcc.gov/). It is worth noting that this is a US regulatory standard, not a global or ITU one. 

A quadkey is assigned a tier based on its test-weighted average download and upload speed for the year (both must clear the tier's thresholds). Population in quadkeys with no observed test activity is reported separately as "No observed activity" rather than folded into "Below minimum".


In [10]:
FCC_TIER_LABELS = [
    "Gigabit (1000/100+ Mbps)",
    "Meets FCC broadband standard (100/20-1000/100 Mbps)",
    "Mid (25/3-100/20 Mbps)",
    "Below minimum (<25/3 Mbps)",
]


def assign_fcc_tier(avg_d_mbps: pd.Series, avg_u_mbps: pd.Series) -> pd.Series:
    """Highest FCC tier for which BOTH download and upload clear that tier's thresholds."""
    conditions = [
        (avg_d_mbps >= 1000) & (avg_u_mbps >= 100),
        (avg_d_mbps >= 100) & (avg_u_mbps >= 20),
        (avg_d_mbps >= 25) & (avg_u_mbps >= 3),
    ]
    choices = FCC_TIER_LABELS[:3]
    return pd.Series(
        np.select(conditions, choices, default=FCC_TIER_LABELS[3]),
        index=avg_d_mbps.index,
    )


def build_speed_tier_population(
    quadkey_year: pd.DataFrame, total_population_by_year: pd.Series
) -> pd.DataFrame:
    active = quadkey_year[quadkey_year["tests"] > 0].copy()
    active["avg_d_mbps"] = active["avg_d_kbps"] / 1000
    active["avg_u_mbps"] = active["avg_u_kbps"] / 1000
    active["tier"] = assign_fcc_tier(active["avg_d_mbps"], active["avg_u_mbps"])

    tier_pop = (
        active.groupby(["type", "year", "tier"], observed=True)["population"]
        .sum()
        .reset_index()
    )

    no_activity = active.groupby(["type", "year"])["population"].sum().reset_index()
    no_activity["total_population"] = no_activity["year"].map(total_population_by_year)
    no_activity["population"] = (
        no_activity["total_population"] - no_activity["population"]
    )
    no_activity = no_activity.drop(columns="total_population")
    no_activity["tier"] = "No observed activity"

    combined = pd.concat([tier_pop, no_activity], ignore_index=True)
    combined["total_population"] = combined["year"].map(total_population_by_year)
    combined["pop_share"] = round(
        combined["population"] / combined["total_population"], 4
    )
    combined = combined.drop(columns="total_population")
    return combined.sort_values(["type", "year"]).reset_index(drop=True)


speed_tier_population = build_speed_tier_population(
    quadkey_year_panel, TOTAL_POPULATION_BY_YEAR
)
speed_tier_population.to_csv(SPEED_TIER_OUT_PATH, index=False)
print(f"Saved -> {SPEED_TIER_OUT_PATH}")
speed_tier_population.pivot_table(
    index=["type", "year"],
    columns="tier",
    values="population",
    fill_value=0,
    observed=True,
)


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/annual_speed_tier_population.csv


tier           Below minimum (<25/3 Mbps)  \
type     year                               
combined 2019                  10526692.9   
         2020                   8582702.4   
         2021                  10749409.5   
         2022                  12351709.3   
         2023                  11952494.9   
         2024                  13321682.5   
         2025                  16983870.0   
fixed    2019                   9909006.8   
         2020                   6943403.9   
         2021                   9374043.0   
         2022                  10932960.4   
         2023                   9437878.2   
         2024                  10848862.0   
         2025                  16508056.1   
mobile   2019                   4669628.0   
         2020                   5337630.9   
         2021                   7015043.3   
         2022                   8952851.2   
         2023                   9338765.4   
         2024                   9577207.0   
         2025                  10824493.0   

tier           Meets FCC broadband standard (100/20-1000/100 Mbps)  \
type     year                                                        
combined 2019                                              156.2     
         2020                                             2101.7     
         2021                                             1121.1     
         2022                                             3819.5     
         2023                                             5587.8     
         2024                                            17905.1     
         2025                                            78648.8     
fixed    2019                                                0.0     
         2020                                             1675.9     
         2021                                             6544.1     
         2022                                             2225.3     
         2023                                             2104.8     
         2024                                            33623.5     
         2025                                           134741.2     
mobile   2019                                             1079.9     
         2020                                              957.6     
         2021                                             3451.1     
         2022                                             4799.0     
         2023                                             5790.1     
         2024                                             8859.6     
         2025                                            67226.8     

tier           Mid (25/3-100/20 Mbps)  No observed activity  
type     year                                                
combined 2019                382600.5             9046737.7  
         2020                737519.7            11428620.6  
         2021                624038.0             9973409.2  
         2022                588870.2             8965325.0  
         2023                931288.8            10125280.3  
         2024               1299867.4             9535187.8  
         2025               3967299.5             4141099.5  
fixed    2019                102008.0             9945172.5  
         2020                134582.0            13671282.6  
         2021                190721.6            11776669.1  
         2022                173570.3            10800968.0  
         2023                274273.9            13300394.9  
         2024                727275.9            12564881.4  
         2025               3654465.3             4873655.2  
mobile   2019               2445553.1            12839926.3  
         2020               2584137.2            12828218.7  
         2021               2258418.4            12071065.0  
         2022               1672950.1            11279123.7  
         2023               2058249.6            11611846.7  
         2024               2955348.7            11

## Quarterly features

In [11]:
QUARTERLY_QUADKEY_OUT_PATH = (
    DATA_DIR / "processed" / "quarterly_quadkey_activity.parquet"
)
QUARTERLY_FEATURES_OUT_PATH = (
    DATA_DIR / "processed" / "quarterly_gdp_proxy_features.csv"
)


def build_quadkey_quarter_panel(
    tiles: pd.DataFrame, population: pd.DataFrame
) -> pd.DataFrame:
    """One row per (type, year, quarter, quadkey): tile speed/latency + population.

    `tiles` is already at this exact grain (each Ookla quarterly file has one
    row per quadkey), so this is a join, not an aggregation. Population is
    joined on `(year, quadkey)` so every quarter uses its own year's
    population layer.
    """
    assert not tiles.duplicated(subset=["type", "year", "quarter", "quadkey"]).any(), (
        "tiles is expected to be unique per (type, year, quarter, quadkey); "
        "if this fires, the merge below silently becomes a fan-out"
    )
    panel = tiles.merge(population, on=["year", "quadkey"], how="left")
    panel["population"] = panel["population"].fillna(0)
    return panel


quadkey_quarter_panel = build_quadkey_quarter_panel(tiles, population)
quadkey_quarter_panel.to_parquet(QUARTERLY_QUADKEY_OUT_PATH, index=False)
print(f"Saved -> {QUARTERLY_QUADKEY_OUT_PATH}")
print(f"{len(quadkey_quarter_panel):,} type-year-quarter-quadkey rows")
quadkey_quarter_panel.head()


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/quarterly_quadkey_activity.parquet
668,623 type-year-quarter-quadkey rows


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,quarter,type,year,population
0,1221103222122032,159.0,2685.0,71.0,4,1,1,fixed,2019,60.4
1,1221103222123001,7401.0,6838.0,28.0,1,1,1,fixed,2019,0.0
2,1221103222131220,809.0,768.0,78.0,1,1,1,fixed,2019,21.7
3,1221103222212232,10816.0,11799.0,35.0,12,1,1,fixed,2019,246.4
4,1221103222230233,4574.0,1974.0,6.0,2,1,1,fixed,2019,97.7


In [12]:
def build_quarterly_national(quadkey_quarter: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (t, y, q), g in quadkey_quarter.groupby(["type", "year", "quarter"]):
        active = g[g["tests"] > 0]
        total_tests = int(active["tests"].sum())
        total_devices = int(active["devices"].sum())
        rows.append(
            dict(
                type=t,
                year=int(y),
                quarter=int(q),
                total_tests=total_tests,
                total_devices=total_devices,
                n_tiles_active=len(active),
                avg_down_mbps=round(
                    weighted_mean(active["avg_d_kbps"], active["tests"]) / 1000, 3
                ),
                avg_up_mbps=round(
                    weighted_mean(active["avg_u_kbps"], active["tests"]) / 1000, 3
                ),
                avg_lat_ms=round(
                    weighted_mean(active["avg_lat_ms"], active["tests"]), 1
                ),
                tests_per_device=round(total_tests / total_devices, 3)
                if total_devices
                else np.nan,
            )
        )
    national = (
        pd.DataFrame(rows)
        .sort_values(["type", "year", "quarter"])
        .reset_index(drop=True)
    )
    national["period"] = pd.to_datetime(
        dict(
            year=national["year"], month=national["quarter"].sub(1).mul(3).add(1), day=1
        )
    )

    return national


quarterly_national = build_quarterly_national(quadkey_quarter_panel)
quarterly_national.to_csv(QUARTERLY_FEATURES_OUT_PATH, index=False)
print(f"Saved -> {QUARTERLY_FEATURES_OUT_PATH}")
quarterly_national.tail()


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/quarterly_gdp_proxy_features.csv


,type,year,quarter,total_tests,total_devices,n_tiles_active,avg_down_mbps,avg_up_mbps,avg_lat_ms,tests_per_device,period
79,mobile,2024,4,27337,13074,4775,19.888,7.443,54.5,2.091,2024-10-01
80,mobile,2025,1,46468,22325,6474,22.635,8.489,61.7,2.081,2025-01-01
81,mobile,2025,2,58893,29767,8146,25.845,8.533,53.5,1.978,2025-04-01
82,mobile,2025,3,44450,24547,7294,33.218,10.381,62.2,1.811,2025-07-01
83,mobile,2025,4,51485,28310,7893,33.747,10.252,64.0,1.819,2025-10-01


## Urban/rural split (GHSL Degree of Urbanisation)

Every population-weighted metric above is a single national number, which can hide very different urban and rural trajectories — e.g. a rising national mean driven entirely by urban recovery while rural areas stagnate or regress. This section classifies each quadkey as urban, rural, or water using the JRC GHSL Settlement Model grid (GHS-SMOD) and re-cuts the annual population and
speed features by that split.

**Data source**: [GHSL GHS-SMOD](https://human-settlement.emergency.copernicus.eu/ghs_smod2023.php), epoch 2020, release R2023A, 30 arcsec (~1km) grid.

**Caveats:**

- GHS-SMOD is ~1km while Ookla zoom-16 quadkey is ~600m x ~490m at Syria's latitude. This is a sample-at-centroid operation (each quadkey takes the SMOD class of the ~1km pixel its centroid falls in), not an area-weighted aggregation — a quadkey straddling an urban/rural boundary gets whichever class its centroid happens to land in. Expect some misclassification right at urban edges; don't over-read small urban/rural gaps as precise.
- Unlike the population layer below (now one raster per year, 2019-2025), GHS-SMOD is only classified once, from the 2020 settlement pattern — a quadkey's urban/rural *label* doesn't change year to year in this notebook, even though the *population* living in that quadkey does. This is a reasonable simplification (built-up-area classification changes far more slowly than population counts), but it does mean any new settlement growth after 2020 is invisible to the urban/rural split.
- Ookla `quadkey`s with `smod_class` in `{11, 12, 13}` are labeled `rural` here; GHSL's own "rural domain" also includes water (`10`) — this pipeline reports water separately instead, since folding water pixels into "rural population" would be misleading (see the water-classified quadkeys in the breakdown below — a small count, but worth not silently merging).


In [13]:
URBANIZATION_PATH = DATA_DIR / "processed" / "quadkey_urbanization.parquet"
URBAN_SPLIT_OUT_PATH = DATA_DIR / "processed" / "annual_urban_rural_features.csv"

urbanization = pd.read_parquet(URBANIZATION_PATH)
print(urbanization["urbanization"].value_counts())

quadkey_year_urb = quadkey_year_panel.merge(urbanization, on="quadkey", how="left")
TOTAL_POPULATION_BY_YEAR_CLASS = (
    population.merge(urbanization, on="quadkey", how="left")
    .groupby(["year", "urbanization"])["population"]
    .sum()
)
TOTAL_POPULATION_BY_YEAR_CLASS.unstack("urbanization")


urbanization
rural    185370
urban     21241
water       118
Name: count, dtype: int64


urbanization,rural,urban,water
year,,,
2019,5079199.0,14857406.4,19581.9
2020,5289550.1,15441009.2,20385.1
2021,5437971.9,15889085.2,20920.7
2022,5574636.8,16313605.1,21482.1
2023,5851769.0,17140305.7,22577.1
2024,6144635.6,18006290.2,23717.0
2025,6397669.1,18748558.8,24689.9


In [14]:
def build_urban_rural_features(
    quadkey_year_urb: pd.DataFrame, total_pop_by_year_class: pd.Series
) -> pd.DataFrame:
    rows = []
    for (t, y, cls), g in quadkey_year_urb.groupby(["type", "year", "urbanization"]):
        if cls not in ("urban", "rural"):
            continue  # skip "water" / unclassified: not meaningful population segments here
        active = g[g["tests"] > 0]
        pop_covered = active["population"].sum()
        total_pop_class = total_pop_by_year_class.get((y, cls), np.nan)
        rows.append(
            dict(
                type=t,
                year=int(y),
                urbanization=cls,
                pop_covered_share=round(pop_covered / total_pop_class, 4)
                if total_pop_class
                else np.nan,
                avg_down_mbps_pop_weighted=round(
                    population_weighted_mean(active["avg_d_kbps"], active["population"])
                    / 1000,
                    3,
                ),
            )
        )
    return (
        pd.DataFrame(rows)
        .sort_values(["type", "urbanization", "year"])
        .reset_index(drop=True)
    )


urban_rural_features = build_urban_rural_features(
    quadkey_year_urb, TOTAL_POPULATION_BY_YEAR_CLASS
)
urban_rural_features.to_csv(URBAN_SPLIT_OUT_PATH, index=False)
print(f"Saved -> {URBAN_SPLIT_OUT_PATH}")
urban_rural_features.pivot_table(
    index=["type", "year"], columns="urbanization", values="pop_covered_share"
)


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/annual_urban_rural_features.csv


urbanization    rural   urban
type     year                
combined 2019  0.2399  0.6511
         2020  0.1107  0.5646
         2021  0.2017  0.6459
         2022  0.2550  0.7052
         2023  0.2301  0.6723
         2024  0.2492  0.7268
         2025  0.5576  0.9302
fixed    2019  0.2171  0.5984
         2020  0.0631  0.4357
         2021  0.1672  0.5446
         2022  0.2199  0.6047
         2023  0.1760  0.5059
         2024  0.1931  0.5777
         2025  0.5261  0.9019
mobile   2019  0.0604  0.4577
         2020  0.0716  0.4883
         2021  0.0917  0.5515
         2022  0.1118  0.6123
         2023  0.1258  0.6212
         2024  0.1442  0.6461
         2025  0.2358  0.7946

## Broadband index

Following the [Esri/ArcGIS Living Atlas "Ookla Speedtest for Global Broadband Performance"](https://www.esri.com/arcgis-blog/products/arcgis-living-atlas/telecommunications/ookla-speedtest-for-global-broadband-performance-in-living-atlas), the broadband index is calculated as follows:

- Each quadkey-year gets a download sub-score and an upload sub-score: `score = (actual speed / benchmark speed) × 100`. A tile at exactly the benchmark scores 100; a tile at double the benchmark scores 200.
- The two sub-scores are only averaged when both clear 100 (both meet the benchmark). If either one is below 100, the total score is the minimum of the two. That means the bottleneck dimension sets the score, rather than a strong download average masking a weak upload (or vice versa). Formally: `total = mean(download_score, upload_score) if both >= 100 else min(download_score, upload_score)`.
- The benchmark matches Esri's methodology (itself built on the current FCC standard), where fixed = 100 Mbps down / 20 Mbps up and mobile = 35 Mbps down / 3 Mbps up.
- Population-weighted mean of the tile-level score up to `type × year` for use as a national annual feature — consistent with how every other
  population-weighted metric in this notebook is aggregated.

Sources: [Esri Living Atlas blog](https://www.esri.com/arcgis-blog/products/arcgis-living-atlas/telecommunications/ookla-speedtest-for-global-broadband-performance-in-living-atlas),
[ArcGIS item page](https://www.arcgis.com/home/item.html?id=048da3d1818b4d0b95ec526b9e642719),
[FCC broadband standard](https://docs.fcc.gov/public/attachments/DOC-401205A1.pdf).


In [15]:
BROADBAND_INDEX_OUT_PATH = DATA_DIR / "processed" / "annual_broadband_index.csv"
BROADBAND_BENCHMARKS_MBPS = {
    "fixed": dict(down=100, up=20),
    "mobile": dict(down=35, up=3),
}


def build_tile_broadband_index(quadkey_year: pd.DataFrame) -> pd.DataFrame:
    df = quadkey_year.copy()
    df["avg_d_mbps"] = df["avg_d_kbps"] / 1000
    df["avg_u_mbps"] = df["avg_u_kbps"] / 1000

    down_benchmark = df["type"].map(lambda t: BROADBAND_BENCHMARKS_MBPS[t]["down"])
    up_benchmark = df["type"].map(lambda t: BROADBAND_BENCHMARKS_MBPS[t]["up"])
    df["download_index"] = 100 * df["avg_d_mbps"] / down_benchmark
    df["upload_index"] = 100 * df["avg_u_mbps"] / up_benchmark

    both_meet_benchmark = (df["download_index"] >= 100) & (df["upload_index"] >= 100)
    df["tile_broadband_index"] = np.where(
        both_meet_benchmark,
        (df["download_index"] + df["upload_index"]) / 2,
        np.minimum(
            df["download_index"], df["upload_index"]
        ),  # weakest link sets the score
    )
    return df


def build_broadband_index(quadkey_year: pd.DataFrame) -> pd.DataFrame:
    # "combined" excluded: benchmarks are technology-specific, and there is no
    # defensible single benchmark for a pooled fixed+mobile connection.
    quadkey_year = quadkey_year[quadkey_year["type"] != "combined"]
    tile_index = build_tile_broadband_index(quadkey_year)
    rows = []
    for (t, y), g in tile_index.groupby(["type", "year"]):
        active = g[g["tests"] > 0]
        rows.append(
            dict(
                type=t,
                year=int(y),
                download_index_pop_weighted=round(
                    population_weighted_mean(
                        active["download_index"], active["population"]
                    ),
                    2,
                ),
                upload_index_pop_weighted=round(
                    population_weighted_mean(
                        active["upload_index"], active["population"]
                    ),
                    2,
                ),
                broadband_index_pop_weighted=round(
                    population_weighted_mean(
                        active["tile_broadband_index"], active["population"]
                    ),
                    2,
                ),
                pop_share_meets_benchmark=round(
                    active.loc[
                        active["tile_broadband_index"] >= 100, "population"
                    ].sum()
                    / active["population"].sum(),
                    4,
                )
                if active["population"].sum() > 0
                else np.nan,
            )
        )
    return pd.DataFrame(rows).sort_values(["type", "year"]).reset_index(drop=True)


broadband_index = build_broadband_index(quadkey_year_panel)
broadband_index.to_csv(BROADBAND_INDEX_OUT_PATH, index=False)
print(f"Saved -> {BROADBAND_INDEX_OUT_PATH}")
broadband_index


Saved -> /Users/farhanreynaldo/Documents/world-bank/git-repo/cross-support-sandbox/data/syria-nowcasting/processed/annual_broadband_index.csv


,type,year,download_index_pop_weighted,upload_index_pop_weighted,broadband_index_pop_weighted,pop_share_meets_benchmark
0,fixed,2019,3.89,19.79,3.83,0.0000
1,fixed,2020,4.11,17.47,4.17,0.0002
2,fixed,2021,5.14,22.88,5.13,0.0007
3,fixed,2022,4.73,22.40,4.69,0.0002
4,fixed,2023,5.47,25.72,5.39,0.0002
5,fixed,2024,9.05,28.09,8.12,0.0029
6,fixed,2025,17.55,43.90,15.95,0.0066
7,mobile,2019,60.59,296.12,82.24,0.1049
8,mobile,2020,57.81,311.79,80.44,0.1010
9,mobile,2021,51.61,305.33,68.03,0.0740
